# Notebook 02 — FastAPI Setup & Webhook Router

## FastAPI คืออะไร?

**FastAPI** คือ Python web framework ที่ใช้รับ HTTP requests
- **Async**: รองรับ `async/await` ทำให้ handle หลาย requests พร้อมกันได้
- **Pydantic**: validate input/output อัตโนมัติ
- **OpenAPI**: สร้าง API docs อัตโนมัติ

---

## โครงสร้าง FastAPI ของโปรเจค

```
src/app/
├── webhook_router.py    ← หลัก: Endpoints & event processing
├── ai_runtime.py        ← Bridge ระหว่าง LINE handler กับ LangGraph
├── app_config.py        ← Constants/defaults
└── helpers/
    └── app_state.py     ← Semaphore สำหรับจำกัด concurrent requests
```


## App Config — `src/app/app_config.py`

ค่า default และ constants ทั้งหมดอยู่ที่นี่:

In [ ]:
import os

# จำลอง app_config.py
# โค้ดจริง: src/app/app_config.py

# LLM defaults
DEFAULT_LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai")
DEFAULT_LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
DEFAULT_LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0"))

# Application settings
LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
MAX_CONCURRENT_EVENTS = int(os.getenv("MAX_CONCURRENT_EVENTS", "1"))

# Data
HOSPITAL_CSV_PATH = os.getenv("HOSPITAL_CSV_PATH", "data/FAQ_hospital_info.csv")

print("Configuration:")
print(f"  LLM Provider: {DEFAULT_LLM_PROVIDER}")
print(f"  LLM Model: {DEFAULT_LLM_MODEL}")
print(f"  Max Concurrent Events: {MAX_CONCURRENT_EVENTS}")
print(f"  Hospital CSV: {HOSPITAL_CSV_PATH}")
print()
print("หมายเหตุ: MAX_CONCURRENT_EVENTS=1 หมายความว่า")
print("  bot จะประมวลผล user ทีละ 1 คน (ป้องกัน race condition ใน session)")

## Lifespan — Startup & Shutdown

FastAPI มี `lifespan` context manager สำหรับ code ที่ต้องรันตอน startup และ cleanup ตอน shutdown

In [ ]:
# จำลอง lifespan ใน webhook_router.py
# โค้ดจริง: src/app/webhook_router.py

from contextlib import asynccontextmanager
import asyncio

@asynccontextmanager
async def lifespan(app):
    """
    รันตอน startup (yield ก่อน) และตอน shutdown (yield หลัง)
    """
    print("=== SERVER STARTUP ===")
    
    # 1. สร้าง Semaphore สำหรับ concurrency control
    # Semaphore = permit ที่จำกัดจำนวน concurrent operations
    max_concurrent = 1  # จาก MAX_CONCURRENT_EVENTS
    app.state.event_semaphore = asyncio.Semaphore(max_concurrent)
    print(f"✓ Created semaphore (max={max_concurrent})")
    
    # 2. โหลด Hospital CSV data เข้า memory
    # init_hospital_index()  ← อ่านจาก data/FAQ_hospital_info.csv
    print("✓ Hospital index loaded")
    
    # 3. Validate Rich Menu env vars
    # _validate_rich_menu_envs()  ← ตรวจสอบว่ามี rich menu IDs
    print("✓ Rich menu envs validated")
    
    yield  # ← Server เริ่มรับ requests ตรงนี้
    
    print("=== SERVER SHUTDOWN ===")
    # cleanup (ถ้าจำเป็น)


# แสดง lifecycle
import asyncio

async def demo_lifespan():
    class MockApp:
        state = type('State', (), {})()
    
    app = MockApp()
    async with lifespan(app):
        print("\n--- Server is running ---")
        print(f"  Semaphore created: {app.state.event_semaphore}")
    print()

asyncio.run(demo_lifespan())

## Webhook Endpoints

FastAPI ในโปรเจคนี้มี 3 endpoints หลัก:

In [ ]:
# จำลอง webhook_router.py
# โค้ดจริง: src/app/webhook_router.py

from fastapi import APIRouter, Request

router = APIRouter()

# ----------------------------------------
# Health Check
# ----------------------------------------
@router.get("/health")
async def health():
    """ตรวจสอบว่า server ยังทำงานอยู่"""
    return {"status": "ok"}


# ----------------------------------------
# DEV Webhook
# ----------------------------------------
@router.post("/webhook-dev")
async def webhook_dev(request: Request):
    """
    รับ webhook จาก LINE DEV account
    1. อ่าน raw body JSON
    2. Loop ผ่าน events (ปกติมี 1 event)
    3. Attach DEV token ไปกับ event
    4. สร้าง async task สำหรับแต่ละ event
    5. Return 200 OK ทันที (ไม่รอให้ AI ตอบ)
    """
    body = await request.json()
    DEV_TOKEN = "dev_channel_access_token"
    
    for event in body.get("events", []):
        event["__line_channel_access_token"] = DEV_TOKEN
        event["__line_account"] = "dev"
        # สร้าง background task — ไม่ block response
        # asyncio.create_task(_process_one_event(request, event))
    
    return {"message": "OK"}


# ----------------------------------------  
# OFFICIAL Webhook
# ----------------------------------------
@router.post("/webhook-official")
async def webhook_official(request: Request):
    """รับ webhook จาก LINE OFFICIAL account (เหมือน dev แต่ใช้ official token)"""
    body = await request.json()
    OFFICIAL_TOKEN = "official_channel_access_token"
    
    for event in body.get("events", []):
        event["__line_channel_access_token"] = OFFICIAL_TOKEN
        event["__line_account"] = "official"
        # asyncio.create_task(_process_one_event(request, event))
    
    return {"message": "OK"}


print("Endpoints ที่ register:")
print("  GET  /health            → health check")
print("  POST /webhook-dev       → LINE dev account")
print("  POST /webhook-official  → LINE official account")
print()
print("สำคัญ! Webhook ต้อง return 200 OK ภายใน 3 วินาที")
print("ดังนั้นจึงใช้ asyncio.create_task() เพื่อ process ใน background")

## Semaphore — Concurrency Control

ปัญหา: ถ้า user ส่งข้อความหลายข้อความพร้อมกัน session จะ race condition

แก้ด้วย **Semaphore**: จำกัดจำนวน events ที่ process พร้อมกัน

In [ ]:
import asyncio
import time

# จำลอง _process_one_event() พร้อม Semaphore
# โค้ดจริง: src/app/webhook_router.py

semaphore = asyncio.Semaphore(1)  # อนุญาตทีละ 1 เท่านั้น

async def process_one_event(event_id: str, event: dict):
    """
    ประมวลผล LINE event 1 ตัว
    ใช้ semaphore เพื่อ serialize concurrent requests จาก user เดียวกัน
    """
    user_id = event.get('user_id', 'unknown')
    
    # acquire semaphore — รอถ้า slot ไม่ว่าง
    async with semaphore:
        print(f"[{event_id}] Processing event for user={user_id}")
        
        # 1. Extract event
        # extracted = extract_line_event(event)
        
        # 2. Show loading animation
        # await show_loading_animation(user_id, token)
        print(f"[{event_id}] Showing loading animation...")
        
        # 3. Run AI processing
        await asyncio.sleep(0.1)  # simulate AI time
        print(f"[{event_id}] AI processing done!")
        
        # 4. Reply to LINE
        # await reply_to_line(reply_token, ...)
        print(f"[{event_id}] Reply sent to LINE")


# ทดสอบ: ส่ง 3 events พร้อมกัน
async def demo_semaphore():
    print("ส่ง 3 events พร้อมกัน (แต่ semaphore=1 → process ทีละ 1):")
    print()
    
    tasks = [
        process_one_event("evt-001", {"user_id": "U123", "text": "message 1"}),
        process_one_event("evt-002", {"user_id": "U123", "text": "message 2"}),
        process_one_event("evt-003", {"user_id": "U123", "text": "message 3"}),
    ]
    
    await asyncio.gather(*tasks)
    print("\nทุก event ถูก process แล้ว (ทีละ 1 เรียงตามลำดับ)")

asyncio.run(demo_semaphore())

## AI Runtime — Bridge Layer

**`src/app/ai_runtime.py`** เป็น bridge ระหว่าง LINE Handler กับ LangGraph

In [ ]:
# จำลอง ai_runtime.py
# โค้ดจริง: src/app/ai_runtime.py

from typing import Any

async def run_ai_runtime(
    session,         # Session object (state + messages)
    user_text: str,  # ข้อความจากผู้ใช้
    postback_data: str = "",  # data จาก postback event
    line_account: str = "dev",
    line_token: str = "",
) -> dict:
    """
    Bridge ระหว่าง LINE handler กับ LangGraph
    
    1. สร้าง state_in จาก session + input
    2. Invoke LangGraph
    3. บันทึก state_out กลับ session
    4. Return HandlerResult
    """
    
    # สร้าง state ส่งเข้า LangGraph
    state_in = {
        # จาก session
        "messages": session.messages[-50:],  # เอาแค่ 50 ข้อความล่าสุด
        "task_status": session.state.get("task_status", "COMPLETED"),
        "conversation_mode": session.state.get("conversation_mode", "AI"),
        # จาก input
        "line_user_id": session.user_id,
        "line_account": line_account,
        "line_channel_access_token": line_token,
        "latest_user_input": user_text,
        "postback_data": postback_data,
    }
    
    print("State ที่ส่งเข้า LangGraph:")
    for k, v in state_in.items():
        if k != 'messages':
            print(f"  {k}: {v!r}")
    print(f"  messages: [{len(state_in['messages'])} messages]")
    
    # # Invoke LangGraph (โค้ดจริง)
    # from src.agent.graph_stub import langgraph_app
    # config = {"configurable": {"thread_id": session.user_id}}
    # state_out = await langgraph_app.ainvoke(state_in, config=config)
    
    # Mock state_out
    state_out = {
        **state_in,
        "final_answer": "นี่คือคำตอบจาก AI",
        "task_status": "COMPLETED",
    }
    
    return state_out


# ทดสอบ
class MockSession:
    user_id = "U123"
    messages = []
    state = {"task_status": "COMPLETED", "conversation_mode": "AI"}

import asyncio
result = asyncio.run(run_ai_runtime(
    session=MockSession(),
    user_text="อยากรู้ข้อมูลโรงพยาบาล",
    line_account="dev"
))
print(f"\nAnswer: {result['final_answer']}")

## Full Request Flow ใน FastAPI

สรุปการเดินทางของ request ตั้งแต่ LINE ถึง response:

In [ ]:
flow_steps = [
    ("1", "LINE Platform",     "POST /webhook-dev",              "raw JSON body"),
    ("2", "webhook_router",    "attach token + create_task()",   "return 200 OK ทันที"),
    ("3", "_process_one_event","acquire semaphore",              "รอถ้าไม่ว่าง"),
    ("4", "messaging.py",      "show_loading_animation()",       "ผู้ใช้เห็นจุด 3 จุด"),
    ("5", "utils.py",          "extract_line_event()",           "normalize event dict"),
    ("6", "handler_stub.py",   "load_session(user_id)",          "โหลด session จาก store"),
    ("7", "ai_runtime.py",     "run_ai_runtime(session, text)",  "เตรียม state_in"),
    ("8", "graph_stub.py",     "langgraph_app.ainvoke(state_in)","AI ประมวลผล"),
    ("9", "handler_stub.py",   "save_session(session)",          "บันทึก state ใหม่"),
    ("10","messaging.py",      "reply_to_line(reply_token)",     "ส่งคำตอบกลับ LINE"),
]

print(f"{'Step':<5} {'Layer':<25} {'Action':<40} {'Result'}")
print("-" * 100)
for step, layer, action, result in flow_steps:
    print(f"{step:<5} {layer:<25} {action:<40} {result}")

## เรียกใช้ FastAPI locally

```bash
# วิธีรัน server
cd AI-Chatbot-LineOA
uvicorn src.app.webhook_router:app --reload --port 8000

# หรือใช้ langgraph dev server
langgraph dev

# ทดสอบ health check
curl http://localhost:8000/health
# → {"status": "ok"}

# ทดสอบ webhook ด้วย ngrok (expose localhost ออก internet)
ngrok http 8000
# จะได้ URL เช่น https://abc123.ngrok.io
# ไปตั้งใน LINE Developer Console:
#   Webhook URL: https://abc123.ngrok.io/webhook-dev
```

**Notebook ถัดไป:** `03_langgraph_basics.ipynb` — เรียนรู้ LangGraph State, Graph, Nodes